Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# Protocol 2 - use: 2 augmented images per original for img1–img4
PROTOCOL2_TRAIN_INDICES = {
    1: [1, 2],
    2: [1, 2],
    3: [1, 2],
    4: [1, 2]
}

# === PARAMETERS FOR (2D)^2PCA ===
NUM_COL_COMPONENTS = 47
NUM_ROW_COMPONENTS = 47

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print(">> Denoising image...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO FUSE FINGERS FOR TRAINING ===
def fuse_fingers_p2(subject_path, subject_id):
    fused_samples = []
    labels = []

    sample_counter = 0

    for img_num, aug_list in PROTOCOL2_TRAIN_INDICES.items():
        for aug_id in aug_list:
            finger_images = []
            complete = True
            sample_counter += 1

            for finger in FINGER_NUMS:
                fname = f"{subject_id}_{finger}_{img_num}_{aug_id}_Augmented.png"
                img_path = os.path.join(subject_path, fname)
                print(f"   >> Loading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"   !! File not found: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                img_denoised = apply_denoising(img, h=10)
                img_eq = exposure.equalize_hist(img_denoised)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                finger_images.append(img_norm)

            if complete and len(finger_images) == 6:
                fused = np.vstack(finger_images)
                fused_samples.append(fused)
                label = f"{subject_id}_fused_p2_{sample_counter:02d}"
                labels.append(label)
                print(f"   >> Created fused sample: {label}")
            else:
                print(f"   !! Skipping incomplete sample for {subject_id}, {img_num}_{aug_id}")

    return fused_samples, labels

# === LOAD TRAINING DATA ===
train_data = []
train_labels = []

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Training Data - Protocol 2 Strategy 1"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n>> Processing subject: {subj}")
    samples, labels = fuse_fingers_p2(subject_path, subj)
    train_data.extend(samples)
    train_labels.extend(labels)

train_data = np.array(train_data)
train_labels = np.array(train_labels)

# === (2D)^2PCA FUNCTIONS ===
def compute_2d2pca(images_2d, num_col_comp=NUM_COL_COMPONENTS, num_row_comp=NUM_ROW_COMPONENTS):
    print("\n>> Computing (2D)^2PCA projection matrices...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n

    G_t = np.zeros((w, w))
    G_r = np.zeros((h, h))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        G_r += A @ A.T
        if i < 3:
            print(f"   >> Sample {i+1} processed")

    G_t /= n
    G_r /= n

    _, V = np.linalg.eigh(G_t)
    _, U = np.linalg.eigh(G_r)

    Wc = V[:, -num_col_comp:]  # column projection
    Wr = U[:, -num_row_comp:]  # row projection

    print(f"   >> Wc shape: {Wc.shape}, Wr shape: {Wr.shape}")
    return Wc, Wr

def project_2d2pca(images_2d, Wc, Wr):
    print("\n📐 Projecting images using (2D)^2PCA...")
    projected = []
    for i, img in enumerate(images_2d):
        feat = Wr.T @ img @ Wc  # Corrected order!
        projected.append(feat)
        if i < 3:
            print(f"   🧮 Projected shape of sample {i + 1}: {feat.shape}")
    return projected

def flatten_projected(projected_imgs):
    print("\n>> Flattening projected features...")
    return np.array([img.flatten() for img in projected_imgs])

# === APPLY (2D)^2PCA ===
Wc, Wr = compute_2d2pca(train_data)
projected_features = project_2d2pca(train_data, Wc, Wr)
flat_features = flatten_projected(projected_features)

print(f"\n✅ Final Feature Matrix Shape: {flat_features.shape}")


Test

In [ ]:
# === TESTING CONFIGURATION FOR PROTOCOL 2 ===
PROTOCOL2_TEST_INDICES = {
    1: ['original', 3],
    2: ['original', 3],
    3: ['original', 3],
    4: ['original', 3]
}

# === FUNCTION TO FUSE FINGERS FOR TESTING ===
def fuse_fingers_p2_test(subject_path, subject_id):
    fused_samples = []
    labels = []

    sample_counter = 0

    for img_num, versions in PROTOCOL2_TEST_INDICES.items():
        for ver in versions:
            finger_images = []
            complete = True
            sample_counter += 1

            if ver == 'original':
                label_tag = f"{img_num}_orig"
            else:
                label_tag = f"{img_num}_{ver}_aug"

            for finger in FINGER_NUMS:
                if ver == 'original':
                    fname = f"{subject_id}_{finger}_{img_num}.png"
                else:
                    fname = f"{subject_id}_{finger}_{img_num}_{ver}_Augmented.png"

                img_path = os.path.join(subject_path, fname)
                print(f"   >> Loading test: {img_path}")

                if not os.path.exists(img_path):
                    print(f"   !! Test file missing: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                img_denoised = apply_denoising(img, h=10)
                img_eq = exposure.equalize_hist(img_denoised)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                finger_images.append(img_norm)

            if complete and len(finger_images) == 6:
                fused = np.vstack(finger_images)
                label = f"{subject_id}_test_{label_tag}"
                fused_samples.append(fused)
                labels.append(label)
                print(f"   >> Created test sample: {label}")
            else:
                print(f"   !! Skipping incomplete test sample {subject_id} {label_tag}")

    return fused_samples, labels

# === LOAD TEST DATA ===
test_data = []
test_labels = []

print("\n📥 Loading Testing Data - Protocol 2 Strategy 1")
for subj in tqdm(subject_dirs, desc="Loading Test Data"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n>> Testing subject: {subj}")
    samples, labels = fuse_fingers_p2_test(subject_path, subj)
    test_data.extend(samples)
    test_labels.extend(labels)

test_data = np.array(test_data)
test_labels = np.array(test_labels)

# === PROJECT TEST DATA ===
projected_test = project_2d2pca(test_data, Wc, Wr)
flat_test_features = flatten_projected(projected_test)

print(f"\n✅ Final Test Feature Shape: {flat_test_features.shape}")
print(f"   >> First 5 test labels: {test_labels[:5]}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n📤 Matching test samples using (2D)^2PCA features (Person ID only)...")

# === Step 1: Compare each test sample to all training samples ===
for i in range(total_tests):
    test_vector = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "0001_test_1_orig"

    # 📏 Compute Manhattan distances to all training vectors
    distances = np.sum(np.abs(flat_features - test_vector), axis=1)

    # 🏆 Nearest neighbor index
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "0001_fused_p2_01"

    # 🎯 Extract only subject IDs
    true_id = true_label.split("_")[0]
    pred_id = predicted_label.split("_")[0]

    # ✅ Person ID match
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
